# ESCOPE: curve-aware arc-like structure detection

This notebook develops and evaluates a classical computer-vision detector for bright, curved, tangential structures in Euclid cutouts. It is an exploratory prioritization tool, not a gravitational-lens classifier.

Compared with the detector previously exposed in ESCOPE, this version adds:

- multiscale local-background subtraction;
- seeded hysteresis segmentation;
- an estimated or manually supplied lens center;
- polar angular-span and radial-thickness measurements;
- tangential-alignment scoring;
- rejection of short straight or border-touching features;
- batch metrics and optional threshold evaluation against manual labels.

The notebook deliberately exposes intermediate images and measurements so thresholds can be calibrated before any detector is re-enabled in the web application.

In [ ]:
%pip install -q opencv-python-headless pandas matplotlib pillow

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import math
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

pd.set_option('display.max_columns', 50)
print('OpenCV:', cv2.__version__)

## Optional Google Drive mount

Set `MOUNT_GOOGLE_DRIVE = True` in Colab when the input cutouts are stored in Drive. The guard avoids remounting an already mounted directory.

In [ ]:
MOUNT_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = '/content/drive'

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    if os.path.ismount(DRIVE_MOUNT_POINT):
        print('Google Drive is already mounted.')
    elif Path(DRIVE_MOUNT_POINT).exists() and any(Path(DRIVE_MOUNT_POINT).iterdir()):
        raise RuntimeError(
            f'{DRIVE_MOUNT_POINT} is not a mount but already contains files. '
            'Use a fresh Colab runtime or choose an empty mount point.'
        )
    else:
        drive.mount(DRIVE_MOUNT_POINT)

## Detector configuration

The initial values are conservative starting points for small Euclid JPEG cutouts. They must be validated on a representative set containing lenses, non-lenses, spiral galaxies, interacting systems, stars, noise, and image artifacts.

In [ ]:
@dataclass(frozen=True)
class ArcDetectorConfig:
    normalize_low_percentile: float = 1.0
    normalize_high_percentile: float = 99.5
    clahe_clip_limit: float = 2.0
    background_sigmas: tuple[float, ...] = (2.0, 4.0, 8.0, 14.0)
    support_percentile: float = 89.0
    seed_percentile: float = 96.5
    min_component_area: int = 5
    min_contour_points: int = 6
    min_area: float = 5.0
    max_area_fraction: float = 0.10
    min_radius_fraction: float = 0.035
    max_radius_fraction: float = 0.55
    min_angular_span_deg: float = 9.0
    min_tangential_alignment: float = 0.52
    min_tangential_ratio: float = 1.35
    max_radial_cv: float = 0.48
    border_margin: int = 2
    score_threshold: float = 0.52
    overlay_alpha: float = 0.55

CONFIG = ArcDetectorConfig()
pd.Series(asdict(CONFIG), name='value').to_frame()

In [ ]:
def robust_normalize(gray: np.ndarray, config: ArcDetectorConfig) -> np.ndarray:
    low, high = np.percentile(
        gray,
        (config.normalize_low_percentile, config.normalize_high_percentile),
    )
    if high <= low:
        return np.zeros_like(gray, dtype=np.uint8)
    normalized = np.clip((gray.astype(np.float32) - low) / (high - low), 0.0, 1.0)
    return np.round(normalized * 255).astype(np.uint8)


def enhance_contrast(gray_normalized: np.ndarray, config: ArcDetectorConfig) -> np.ndarray:
    clahe = cv2.createCLAHE(
        clipLimit=config.clahe_clip_limit,
        tileGridSize=(8, 8),
    )
    return clahe.apply(gray_normalized)


def multiscale_positive_residual(
    gray_equalized: np.ndarray,
    config: ArcDetectorConfig,
) -> np.ndarray:
    source = gray_equalized.astype(np.float32)
    responses = []
    for sigma in config.background_sigmas:
        background = cv2.GaussianBlur(source, (0, 0), sigmaX=sigma, sigmaY=sigma)
        responses.append(np.maximum(source - background, 0.0))
    residual = np.max(np.stack(responses), axis=0)
    high = float(np.percentile(residual, 99.7))
    if high <= 0:
        return np.zeros_like(gray_equalized, dtype=np.uint8)
    return np.round(np.clip(residual / high, 0.0, 1.0) * 255).astype(np.uint8)


def estimate_lens_center(gray_equalized: np.ndarray) -> tuple[float, float]:
    height, width = gray_equalized.shape
    y_grid, x_grid = np.mgrid[:height, :width]
    image_center = np.array([(width - 1) / 2.0, (height - 1) / 2.0])
    prior_sigma = 0.22 * min(height, width)
    prior = np.exp(
        -((x_grid - image_center[0]) ** 2 + (y_grid - image_center[1]) ** 2)
        / (2.0 * prior_sigma**2)
    )
    smooth = cv2.GaussianBlur(gray_equalized.astype(np.float32), (0, 0), 3.0)
    weighted = smooth * prior
    peak_y, peak_x = np.unravel_index(np.argmax(weighted), weighted.shape)

    local_radius = max(4, int(round(0.10 * min(height, width))))
    local = (x_grid - peak_x) ** 2 + (y_grid - peak_y) ** 2 <= local_radius**2
    weights = np.maximum(smooth - np.percentile(smooth[local], 40), 0.0) * local
    total_weight = float(weights.sum())
    if total_weight <= 0:
        return float(peak_x), float(peak_y)
    center_x = float((weights * x_grid).sum() / total_weight)
    center_y = float((weights * y_grid).sum() / total_weight)
    return center_x, center_y


def seeded_hysteresis_mask(
    residual: np.ndarray,
    gray_equalized: np.ndarray,
    config: ArcDetectorConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    support_threshold = float(np.percentile(residual, config.support_percentile))
    seed_threshold = float(np.percentile(residual, config.seed_percentile))
    support = residual >= support_threshold
    seeds = residual >= seed_threshold

    median = float(np.median(gray_equalized))
    lower = int(max(0, 0.66 * median))
    upper = int(min(255, max(lower + 1, 1.33 * median)))
    edges = cv2.Canny(gray_equalized, lower, upper) > 0
    edge_gate = cv2.dilate(edges.astype(np.uint8), np.ones((3, 3), np.uint8), 1) > 0
    middle_threshold = 0.5 * (support_threshold + seed_threshold)
    support &= edge_gate | (residual >= middle_threshold)

    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        support.astype(np.uint8),
        connectivity=8,
    )
    retained = np.zeros_like(support, dtype=np.uint8)
    for label_id in range(1, count):
        component = labels == label_id
        if stats[label_id, cv2.CC_STAT_AREA] < config.min_component_area:
            continue
        if np.any(seeds[component]):
            retained[component] = 255

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    retained = cv2.morphologyEx(retained, cv2.MORPH_CLOSE, kernel, iterations=1)
    return retained, edges.astype(np.uint8) * 255, seeds.astype(np.uint8) * 255


def minimal_angular_span_radians(angles: np.ndarray) -> float:
    if len(angles) < 2:
        return 0.0
    ordered = np.sort(np.mod(angles, 2.0 * np.pi))
    gaps = np.diff(np.concatenate([ordered, ordered[:1] + 2.0 * np.pi]))
    return float(2.0 * np.pi - gaps.max())

In [ ]:
def contour_measurements(
    contour: np.ndarray,
    residual: np.ndarray,
    center_xy: tuple[float, float],
    config: ArcDetectorConfig,
) -> dict[str, float | bool | str | np.ndarray]:
    height, width = residual.shape
    points = contour[:, 0, :].astype(np.float64)
    area = float(cv2.contourArea(contour))
    perimeter = float(cv2.arcLength(contour, closed=True))
    x, y, box_width, box_height = cv2.boundingRect(contour)
    touches_border = (
        x <= config.border_margin
        or y <= config.border_margin
        or x + box_width >= width - config.border_margin
        or y + box_height >= height - config.border_margin
    )

    centroid = points.mean(axis=0)
    centered_points = points - centroid
    covariance = np.cov(centered_points, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    eigenvalues = np.maximum(eigenvalues, 1e-6)
    major_axis = eigenvectors[:, int(np.argmax(eigenvalues))]
    elongation = float(np.sqrt(eigenvalues.max() / eigenvalues.min()))

    lens_center = np.asarray(center_xy, dtype=np.float64)
    radial_vector = centroid - lens_center
    centroid_radius = float(np.linalg.norm(radial_vector))
    if centroid_radius > 1e-6:
        radial_unit = radial_vector / centroid_radius
        tangent_unit = np.array([-radial_unit[1], radial_unit[0]])
        tangential_alignment = float(abs(np.dot(major_axis, tangent_unit)))
    else:
        tangential_alignment = 0.0

    relative = points - lens_center
    radii = np.linalg.norm(relative, axis=1)
    angles = np.arctan2(relative[:, 1], relative[:, 0])
    angular_span = minimal_angular_span_radians(angles)
    angular_span_deg = float(np.degrees(angular_span))
    radius_mean = float(np.mean(radii))
    radial_thickness = float(max(np.percentile(radii, 90) - np.percentile(radii, 10), 1.0))
    radial_cv = float(np.std(radii) / max(radius_mean, 1e-6))
    tangential_extent = float(radius_mean * angular_span)
    tangential_ratio = float(tangential_extent / radial_thickness)

    contour_mask = np.zeros_like(residual, dtype=np.uint8)
    cv2.drawContours(contour_mask, [contour], -1, 255, thickness=cv2.FILLED)
    inside_values = residual[contour_mask > 0]
    residual_strength = float(np.mean(inside_values) / 255.0) if len(inside_values) else 0.0

    image_scale = float(min(height, width))
    radius_fraction = centroid_radius / image_scale
    area_fraction = area / float(height * width)

    span_score = np.clip(angular_span_deg / 75.0, 0.0, 1.0)
    alignment_score = np.clip((tangential_alignment - 0.4) / 0.6, 0.0, 1.0)
    thin_arc_score = np.clip((tangential_ratio - 1.0) / 4.0, 0.0, 1.0)
    radial_consistency_score = np.clip(1.0 - radial_cv / config.max_radial_cv, 0.0, 1.0)
    strength_score = np.clip((residual_strength - 0.20) / 0.55, 0.0, 1.0)
    score = float(
        0.24 * span_score
        + 0.25 * alignment_score
        + 0.22 * thin_arc_score
        + 0.19 * radial_consistency_score
        + 0.10 * strength_score
    )

    rejection_reasons = []
    if len(contour) < config.min_contour_points:
        rejection_reasons.append('few_contour_points')
    if area < config.min_area:
        rejection_reasons.append('small_area')
    if area_fraction > config.max_area_fraction:
        rejection_reasons.append('large_area')
    if not config.min_radius_fraction <= radius_fraction <= config.max_radius_fraction:
        rejection_reasons.append('radius')
    if angular_span_deg < config.min_angular_span_deg:
        rejection_reasons.append('short_angular_span')
    if tangential_alignment < config.min_tangential_alignment:
        rejection_reasons.append('not_tangential')
    if tangential_ratio < config.min_tangential_ratio:
        rejection_reasons.append('too_thick_or_short')
    if radial_cv > config.max_radial_cv:
        rejection_reasons.append('not_concentric')
    if touches_border:
        rejection_reasons.append('touches_border')
    if elongation > 5.0 and angular_span_deg < 16.0:
        rejection_reasons.append('straight_feature')
    if score < config.score_threshold:
        rejection_reasons.append('low_score')

    return {
        'contour': contour,
        'centroid_x': float(centroid[0]),
        'centroid_y': float(centroid[1]),
        'area': area,
        'perimeter': perimeter,
        'elongation': elongation,
        'radius_fraction': radius_fraction,
        'angular_span_deg': angular_span_deg,
        'tangential_alignment': tangential_alignment,
        'tangential_ratio': tangential_ratio,
        'radial_cv': radial_cv,
        'residual_strength': residual_strength,
        'score': score,
        'accepted': not rejection_reasons,
        'rejection_reason': ','.join(rejection_reasons),
    }


def alpha_overlay(rgb: np.ndarray, mask: np.ndarray, alpha: float) -> np.ndarray:
    output = rgb.astype(np.float32).copy()
    selected = mask > 0
    red = np.zeros_like(output)
    red[..., 0] = 255
    output[selected] = (1.0 - alpha) * output[selected] + alpha * red[selected]
    return np.clip(output, 0, 255).astype(np.uint8)


def detect_arc_like_structures(
    image_path: str | Path,
    config: ArcDetectorConfig = CONFIG,
    center_xy: tuple[float, float] | None = None,
) -> dict[str, object]:
    image_path = Path(image_path)
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f'Could not read image: {image_path}')

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    normalized = robust_normalize(gray, config)
    equalized = enhance_contrast(normalized, config)
    residual = multiscale_positive_residual(equalized, config)
    binary, edges, seeds = seeded_hysteresis_mask(residual, equalized, config)
    if center_xy is None:
        center_xy = estimate_lens_center(equalized)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    measured = [
        contour_measurements(contour, residual, center_xy, config)
        for contour in contours
        if len(contour) >= config.min_contour_points
    ]
    measured.sort(key=lambda row: float(row['score']), reverse=True)
    for candidate_index, row in enumerate(measured):
        row['candidate_index'] = candidate_index
    accepted = [row for row in measured if bool(row['accepted'])]

    candidate_mask = np.zeros_like(gray, dtype=np.uint8)
    for row in accepted:
        cv2.drawContours(candidate_mask, [row['contour']], -1, 255, cv2.FILLED)
    overlay = alpha_overlay(rgb, candidate_mask, config.overlay_alpha)
    contour_preview = rgb.copy()
    for row in measured:
        color = (255, 0, 0) if row['accepted'] else (0, 190, 255)
        cv2.drawContours(contour_preview, [row['contour']], -1, color, 1)
        cv2.putText(
            contour_preview,
            str(row['candidate_index']),
            (int(round(row['centroid_x'])), int(round(row['centroid_y']))),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.32,
            color,
            1,
            cv2.LINE_AA,
        )
    center_preview = overlay.copy()
    cv2.drawMarker(
        center_preview,
        (int(round(center_xy[0])), int(round(center_xy[1]))),
        (0, 255, 255),
        markerType=cv2.MARKER_CROSS,
        markerSize=9,
        thickness=1,
    )

    metric_columns = [
        'candidate_index', 'centroid_x', 'centroid_y', 'area', 'perimeter',
        'elongation', 'radius_fraction',
        'angular_span_deg', 'tangential_alignment', 'tangential_ratio',
        'radial_cv', 'residual_strength', 'score', 'accepted',
        'rejection_reason',
    ]
    metrics = pd.DataFrame(
        [{key: row[key] for key in metric_columns} for row in measured]
    )
    return {
        'path': image_path,
        'original': rgb,
        'normalized': normalized,
        'equalized': equalized,
        'residual': residual,
        'edges': edges,
        'seeds': seeds,
        'binary': binary,
        'mask': candidate_mask,
        'contour_preview': contour_preview,
        'overlay': overlay,
        'center_preview': center_preview,
        'center_xy': center_xy,
        'metrics': metrics,
        'n_candidates': len(accepted),
        'max_score': max((float(row['score']) for row in measured), default=0.0),
    }

In [ ]:
def show_diagnostics(result: dict[str, object], max_metric_rows: int = 30) -> None:
    panels = [
        ('Original', result['original'], None),
        ('Robust normalization', result['normalized'], 'gray'),
        ('CLAHE', result['equalized'], 'gray'),
        ('Multiscale residual', result['residual'], 'magma'),
        ('Canny edges', result['edges'], 'gray'),
        ('Hysteresis mask', result['binary'], 'gray'),
        ('Scored contours: red=accepted', result['contour_preview'], None),
        ('Accepted arc mask', result['mask'], 'gray'),
        ('Overlay and estimated center', result['center_preview'], None),
    ]
    figure, axes = plt.subplots(3, 3, figsize=(15, 15))
    for axis, (title, image, cmap) in zip(axes.flat, panels):
        axis.imshow(image, cmap=cmap)
        axis.set_title(title)
        axis.axis('off')
    figure.suptitle(
        f"{Path(result['path']).name} | accepted={result['n_candidates']} | "
        f"max score={result['max_score']:.3f}",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()

    metrics = result['metrics']
    if metrics.empty:
        print('No connected contour candidates were found.')
    else:
        display(metrics.head(max_metric_rows).style.format(precision=3))

## Single-image diagnostic

Upload a cutout to Colab or point `IMAGE_PATH` to Google Drive. If the automatic center is incorrect, set `MANUAL_CENTER_XY = (x, y)` and rerun. Center estimation is a major source of error for off-center or multi-deflector systems.

In [ ]:
IMAGE_PATH = '/content/lens004.jpeg'
MANUAL_CENTER_XY = None  # Example: (64.0, 63.0)

single_result = detect_arc_like_structures(
    IMAGE_PATH,
    config=CONFIG,
    center_xy=MANUAL_CENTER_XY,
)
show_diagnostics(single_result)

## Limited batch evaluation

This section is disabled by default. Start with a small, representative validation folder instead of the complete catalogue. It writes one row per image and saves overlays only for images with accepted candidates.

In [ ]:
RUN_BATCH = False
IMAGE_DIRECTORY = Path('/content/arc_validation_images')
IMAGE_PATTERN = '*.jpg'
MAX_BATCH_IMAGES = 200
BATCH_OUTPUT_DIRECTORY = Path('/content/arc_detection_v2_output')

batch_rows = []
if RUN_BATCH:
    BATCH_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    overlay_directory = BATCH_OUTPUT_DIRECTORY / 'positive_overlays'
    overlay_directory.mkdir(parents=True, exist_ok=True)
    image_paths = sorted(IMAGE_DIRECTORY.glob(IMAGE_PATTERN))[:MAX_BATCH_IMAGES]
    print(f'Processing {len(image_paths)} images')

    for index, image_path in enumerate(image_paths, start=1):
        try:
            result = detect_arc_like_structures(image_path, CONFIG)
            batch_rows.append({
                'file_name': image_path.name,
                'path': str(image_path),
                'n_candidates': result['n_candidates'],
                'max_score': result['max_score'],
                'center_x': result['center_xy'][0],
                'center_y': result['center_xy'][1],
                'error': '',
            })
            if result['n_candidates'] > 0:
                output_path = overlay_directory / image_path.name
                Image.fromarray(result['overlay']).save(output_path, quality=92)
        except Exception as exc:
            batch_rows.append({
                'file_name': image_path.name,
                'path': str(image_path),
                'n_candidates': 0,
                'max_score': 0.0,
                'center_x': np.nan,
                'center_y': np.nan,
                'error': str(exc),
            })
        if index % 25 == 0:
            print(f'{index}/{len(image_paths)}')

    batch_metrics = pd.DataFrame(batch_rows)
    batch_metrics.to_csv(BATCH_OUTPUT_DIRECTORY / 'image_scores.csv', index=False)
    display(batch_metrics.sort_values('max_score', ascending=False).head(50))
else:
    print('Batch processing is disabled. Set RUN_BATCH = True after configuring the paths.')

## Optional manual-label evaluation

Create a CSV with columns `file_name` and `has_arc`, where `has_arc` is 1 for a visually supported arc-like structure and 0 otherwise. This evaluates image-level ranking using the maximum candidate score. Do not tune and report performance on the same images; keep a separate test set.

In [ ]:
MANUAL_LABELS_CSV = ''

if MANUAL_LABELS_CSV and batch_rows:
    labels = pd.read_csv(MANUAL_LABELS_CSV)
    scores = pd.DataFrame(batch_rows).merge(labels, on='file_name', how='inner')
    evaluation_rows = []
    for threshold in np.linspace(0.20, 0.90, 71):
        predicted = scores['max_score'] >= threshold
        actual = scores['has_arc'].astype(bool)
        true_positive = int((predicted & actual).sum())
        false_positive = int((predicted & ~actual).sum())
        false_negative = int((~predicted & actual).sum())
        precision = true_positive / max(true_positive + false_positive, 1)
        recall = true_positive / max(true_positive + false_negative, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        evaluation_rows.append({
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'true_positive': true_positive,
            'false_positive': false_positive,
            'false_negative': false_negative,
        })
    evaluation = pd.DataFrame(evaluation_rows)
    display(evaluation.sort_values(['f1', 'precision'], ascending=False).head(15))
    evaluation.plot(x='threshold', y=['precision', 'recall', 'f1'], ylim=(0, 1), figsize=(9, 5))
    plt.grid(alpha=0.25)
    plt.show()
else:
    print('Provide MANUAL_LABELS_CSV and run the batch section to evaluate thresholds.')

## Validation checklist before ESCOPE integration

1. Build a balanced validation set covering A/B/C candidates and difficult unknown objects.
2. Label whether a visible arc-like structure is present independently of the catalogue grade.
3. Inspect center-estimation failures separately.
4. Select thresholds using a development set and report precision/recall on a held-out test set.
5. Review false positives from spiral arms, tidal features, edge-on galaxies, diffraction spikes, and straight artifacts.
6. Re-enable the ESCOPE control only when the overlay is scientifically useful across the validation set, not just on selected examples.